# 04 Feature Engineering

This notebook creates the target variable for inspection likelihood and adds temporal and trend-based features used for modeling.

In [10]:
%pip install --quiet scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Author: Jessica
# Loading inspection-level data so this notebook runs independently

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

inspection_level = pd.read_csv(
    "shared_data/inspection_level.csv",
    parse_dates=['INSPECTION DATE']
)
print(inspection_level.shape)
inspection_level.head()

(74577, 22)


,CAMIS,INSPECTION DATE,score,grade,total_violations,critical_violations,noncritical_violations,unique_categories,critical_ratio,has_critical,...,viol_facility conditions_count,viol_food handling_count,viol_food temperature_count,viol_other_count,viol_personal hygiene_count,viol_pest control_count,viol_trans fat_count,historical_avg_score,historical_avg_critical,prior_inspection_count
0,30075445,2023-08-01,38.0,NaN,3,2,1,2,0.666667,1,...,1,2,0,0,0,0,0,NaN,NaN,0
1,30075445,2023-08-22,12.0,A,3,1,2,2,0.333333,1,...,2,1,0,0,0,0,0,38.0,2.0,1
2,30075445,2024-11-08,10.0,A,3,1,2,3,0.333333,1,...,1,0,0,0,1,1,0,25.0,1.5,2
3,30191841,2023-04-23,10.0,A,2,2,0,1,1.000000,1,...,0,0,0,0,2,0,0,NaN,NaN,0
4,30191841,2024-11-20,24.0,NaN,6,2,4,4,0.333333,1,...,3,1,0,0,1,1,0,10.0,2.0,1


In [4]:
# Author: Jessica
# Creating a target variable to predict if a restaurant gets inspected again within 90 days

# getting next inspection date for each restaurant
inspection_level['next_inspection_date'] = (
    inspection_level.groupby('CAMIS')['INSPECTION DATE'].shift(-1)
)

# calculating how many days until the next inspection
inspection_level['days_until_next_inspection'] = (
    inspection_level['next_inspection_date'] - inspection_level['INSPECTION DATE']
).dt.days

# creating binary target (1 = inspected within 90 days, 0 = not)
inspection_level['inspection_within_90_days'] = (
    inspection_level['days_until_next_inspection'] <= 90
).astype(int)

# dropping rows where we don’t have a future inspection
inspection_model_df = inspection_level.dropna(subset=['next_inspection_date']).copy()

In [5]:
# Author: Jessica
# Adding time-based and trend features to help capture inspection patterns

inspection_model_df['inspection_month'] = inspection_model_df['INSPECTION DATE'].dt.month
inspection_model_df['inspection_year'] = inspection_model_df['INSPECTION DATE'].dt.year

# calculating how many days have passed since the last inspection
inspection_model_df['days_since_last_inspection'] = (
    inspection_model_df.groupby('CAMIS')['INSPECTION DATE'].diff().dt.days
)

# getting previous score for each restaurant
inspection_model_df['previous_score'] = (
    inspection_model_df.groupby('CAMIS')['score'].shift(1)
)

# calculating how much the score changed from the previous inspection
inspection_model_df['score_change_from_previous'] = (
    inspection_model_df['score'] - inspection_model_df['previous_score']
)

In [6]:
#check 
inspection_model_df[[
    'CAMIS',
    'INSPECTION DATE',
    'inspection_month',
    'inspection_year',
    'days_since_last_inspection',
    'previous_score',
    'score_change_from_previous'
]].head(10)

,CAMIS,INSPECTION DATE,inspection_month,inspection_year,days_since_last_inspection,previous_score,score_change_from_previous
0,30075445,2023-08-01,8,2023,NaN,NaN,NaN
1,30075445,2023-08-22,8,2023,21.0,38.0,-26.0
3,30191841,2023-04-23,4,2023,NaN,NaN,NaN
4,30191841,2024-11-20,11,2024,577.0,10.0,14.0
6,40356018,2024-04-16,4,2024,NaN,NaN,NaN
8,40356483,2022-01-24,1,2022,NaN,NaN,NaN
9,40356483,2022-08-03,8,2022,191.0,9.0,12.0
10,40356483,2022-08-19,8,2022,16.0,21.0,-19.0
11,40356483,2023-11-16,11,2023,454.0,2.0,33.0
12,40356483,2024-04-23,4,2024,159.0,35.0,-8.0


In [7]:
#Author: Joel Barnes
#Adding 'time since open' feature

# Assuming restaurants open 5 years before their first inspection
first_inspection_dates = inspection_level.groupby('CAMIS')['INSPECTION DATE'].min()
assumed_open_dates = first_inspection_dates - pd.DateOffset(years=5)

inspection_model_df['assumed_open_date'] = inspection_model_df['CAMIS'].map(assumed_open_dates)
inspection_model_df['time_since_open_days'] = (inspection_model_df['INSPECTION DATE'] - inspection_model_df['assumed_open_date']).dt.days

In [8]:

#check 
inspection_model_df[[
    'CAMIS',
    'INSPECTION DATE',
    'inspection_month',
    'inspection_year',
    'days_since_last_inspection',
    'previous_score',
    'score_change_from_previous',
    'assumed_open_date',
    'time_since_open_days'
]].head(10)

,CAMIS,INSPECTION DATE,inspection_month,inspection_year,days_since_last_inspection,previous_score,score_change_from_previous,assumed_open_date,time_since_open_days
0,30075445,2023-08-01,8,2023,NaN,NaN,NaN,2018-08-01,1826
1,30075445,2023-08-22,8,2023,21.0,38.0,-26.0,2018-08-01,1847
3,30191841,2023-04-23,4,2023,NaN,NaN,NaN,2018-04-23,1826
4,30191841,2024-11-20,11,2024,577.0,10.0,14.0,2018-04-23,2403
6,40356018,2024-04-16,4,2024,NaN,NaN,NaN,2019-04-16,1827
8,40356483,2022-01-24,1,2022,NaN,NaN,NaN,2017-01-24,1826
9,40356483,2022-08-03,8,2022,191.0,9.0,12.0,2017-01-24,2017
10,40356483,2022-08-19,8,2022,16.0,21.0,-19.0,2017-01-24,2033
11,40356483,2023-11-16,11,2023,454.0,2.0,33.0,2017-01-24,2487
12,40356483,2024-04-23,4,2024,159.0,35.0,-8.0,2017-01-24,2646


In [ ]:
# Author: Zeek and Mian
# Adding geographic and socioeconomic features to inspection_model_df
# Columns boro, zipcode, latitude, longitude already exist via notebook 03
# This cell goes AFTER Joel's 'time since open' cell and BEFORE Jessica's save cell
 
# ── Step 1: Clean lat/long values ───────────────────────────────────────────
# Some records have 0 for coordinates — treat those as missing and impute
# with borough medians so the spatial grid doesn't get polluted.
 
inspection_model_df['latitude'] = inspection_model_df['latitude'].replace(0, np.nan)
inspection_model_df['longitude'] = inspection_model_df['longitude'].replace(0, np.nan)
 
inspection_model_df['latitude'] = inspection_model_df.groupby('boro')['latitude'].transform(
lambda x: x.fillna(x.median())
)
inspection_model_df['longitude'] = inspection_model_df.groupby('boro')['longitude'].transform(
lambda x: x.fillna(x.median())
)
 
# ── Step 2: Borough encoding ────────────────────────────────────────────────
# One-hot encode boro so the model can distinguish between boroughs.
 
boro_dummies = pd.get_dummies(inspection_model_df['boro'], prefix='boro')
inspection_model_df = pd.concat([inspection_model_df, boro_dummies], axis=1)
 
# ── Step 3: Spatial grid from lat/long ──────────────────────────────────────
# Bin coordinates into a grid so the model learns location patterns
# without overfitting to exact coordinates.
 
N_BINS = 25
inspection_model_df['lat_bin'] = pd.cut(inspection_model_df['latitude'], bins=N_BINS, labels=False)
inspection_model_df['lon_bin'] = pd.cut(inspection_model_df['longitude'], bins=N_BINS, labels=False)
inspection_model_df['grid_cell'] = (
inspection_model_df['lat_bin'].astype(str) + '_' + inspection_model_df['lon_bin'].astype(str)
)
 
# ── Step 4: Zipcode-level proxy features ────────────────────────────────────
# Neighborhood-quality proxies derived from the inspection data itself.
 
# Average inspection score per zipcode
zip_avg_score = (
inspection_model_df
.groupby('zipcode')['score']
.mean()
.rename('zip_avg_score')
)
inspection_model_df = inspection_model_df.merge(zip_avg_score, on='zipcode', how='left')
 
# Restaurant density per zipcode
zip_restaurant_count = (
inspection_model_df
.groupby('zipcode')['CAMIS']
.nunique()
.rename('zip_restaurant_count')
)
inspection_model_df = inspection_model_df.merge(zip_restaurant_count, on='zipcode', how='left')
 
# ── Step 5: Borough-level socioeconomic data (Census/ACS estimates) ─────────
 
borough_socioeconomic = pd.DataFrame({
'boro': ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'],
'median_household_income': [93651, 67940, 75748, 40935, 85381],
'population_density_per_sq_mi': [72918, 39438, 21460, 34653, 8618],
'poverty_rate_pct': [14.3, 19.2, 11.4, 27.3, 10.7]
})
 
inspection_model_df = inspection_model_df.merge(borough_socioeconomic, on='boro', how='left')
 
# ── Step 6: Inspection coverage ratio ───────────────────────────────────────
# Key feature for surfacing under-inspected areas.
# A low ratio = many restaurants but few inspections in that zipcode.
 
zip_inspection_count = (
inspection_model_df
.groupby('zipcode')['CAMIS']
.count()
.rename('zip_total_inspections')
)
inspection_model_df = inspection_model_df.merge(zip_inspection_count, on='zipcode', how='left')
 
inspection_model_df['zip_inspections_per_restaurant'] = (
inspection_model_df['zip_total_inspections'] / inspection_model_df['zip_restaurant_count']
)
 
# ── Step 7: Sanity check ───────────────────────────────────────────────────
new_cols = [
'boro', 'zipcode', 'latitude', 'longitude', 'grid_cell',
'zip_avg_score', 'zip_restaurant_count',
'median_household_income', 'population_density_per_sq_mi', 'poverty_rate_pct',
'zip_inspections_per_restaurant'
]
print("New feature sample:")
print(inspection_model_df[['CAMIS', 'INSPECTION DATE'] + new_cols].head(10))
print(f"\nDataframe shape after geo/socioeconomic features: {inspection_model_df.shape}")

In [9]:
# Author: Jessica
# Saving model-ready data so later notebooks can run independently

model_df_path = "shared_data/inspection_model_df.csv"
inspection_model_df.to_csv(model_df_path, index=False)
print(f"Saved model-ready dataset to {model_df_path}")

Saved model-ready dataset to shared_data/inspection_model_df.csv
